# SQL Experiment Orchestrator

This notebook orchestrates the 2-stage experiment workflow:
1. **Generation (`run_experiments.py`)**: Runs LLMs to generate SQL.
2. **Analysis (`analyze_results.py`)**: Aggregates results, runs equivalence evaluation, and generates plots.

**Local Execution**: Runs entirely on the Colab VM for speed, backing up to Drive at the end.

In [ ]:
# Cell 1: Setup Infrastructure
import os
import shutil
import zipfile
from google.colab import drive
from google.colab import userdata

# 1. Environment Config
LOCAL_BASE = '/content'
DRIVE_BASE = '/content/drive/MyDrive/ExpResults'
REPO_PATH = f'{LOCAL_BASE}/sql-nl'
ZIP_PATH = '/content/exp.zip'

# 2. Mount Drive
drive.mount('/content/drive')

# 3. Install Dependencies
!pip install -q vllm openai anthropic google-genai sqlglot tqdm pandas matplotlib seaborn pyyaml python-dotenv huggingface_hub

# 4. Authentication (Secrets)
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print("✅ Auth successful")
except Exception as e:
    print(f"⚠️ Auth warning: {e}")

# 5. Unpack Codebase
if os.path.exists(LOCAL_BASE):
    shutil.rmtree(LOCAL_BASE)
os.makedirs(LOCAL_BASE, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError("❌ exp.zip not found! Please upload it.")

print("📦 Unzipping code...")
!unzip exp.zip -d ./
    
print("✅ Setup Complete")

In [ ]:
# Cell 2: Run Generation Phase
# Runs run_experiments.py
# This step generates SQL for the models configured in the script.

import sys
# Set env vars for the script
os.environ['LOCAL_BASE'] = LOCAL_BASE
os.environ['REPO_PATH'] = REPO_PATH
os.environ['DRIVE_BASE'] = DRIVE_BASE

# Run script
!python {REPO_PATH}/run_experiments.py

## Analaysis when the experiment is running

In [3]:
# Cell 3: Run Analysis Phase
# Runs analyze_results.py
# Finds the latest run directory and processes it.

import glob
import os
# Find latest run dir
# runs_dir = f'{LOCAL_BASE}/runs'
runs_dir = f'sample_exp_run/20each/output'
list_of_runs = glob.glob(f'{runs_dir}/*')
latest_run = max(list_of_runs, key=os.path.getctime)
outputs_dir = f'{latest_run}/outputs'

print(f"🔍 Analyzing results in: {outputs_dir}")

# !python {REPO_PATH}/analyze_results.py {outputs_dir}

!python analyze_results.py {outputs_dir}

🔍 Analyzing results in: sample_exp_run/20each/output/results_local-qwen3-coder-30b-a3b_20260204_225948.jsonl/outputs
📂 Copying test databases to local VM...
✅ Database files copied.
⚠️ No result files found in sample_exp_run/20each/output/results_local-qwen3-coder-30b-a3b_20260204_225948.jsonl/outputs


## Analysis when using static jsonl files collected in one place

In [ ]:
# Cell 5: Manual Local Analysis (Folder)
# Use this cell to analyze a specific folder of results locally.

# --- CONFIGURATION ---
# Path to the folder containing your .jsonl files
# Example: '/Users/obby/Downloads/my_experiment_results'
TARGET_OUTPUT_DIR = '/Users/obby/Documents/experiment/random/sql-nl/sample_exp_run/20each/output'
# ---------------------

import os

if not os.path.exists(TARGET_OUTPUT_DIR):
    print(f"❌ Directory not found: {TARGET_OUTPUT_DIR}")
else:
    print(f"🚀 Starting analysis in: {TARGET_OUTPUT_DIR}")
    
    # Run analysis script pointing to the directory
    # Assuming the script is in the current directory ('.')
    !python analyze_results.py "{TARGET_OUTPUT_DIR}"

🚀 Starting analysis in: /Users/obby/Documents/experiment/random/sql-nl/sample_exp_run/20each/output
📂 Copying test databases to local VM...
✅ Database files copied.
📋 Found 6 result files in /Users/obby/Documents/experiment/random/sql-nl/sample_exp_run/20each/output
✅ Aggregated 19,578 total records.
running evaluation...
Evaluating SQL: 100%|█████████████████████| 19578/19578 [41:31<00:00,  7.86it/s]
💾 Evaluated results saved to /Users/obby/Documents/experiment/random/sql-nl/sample_exp_run/20each/output/evaluated_results_aggregated.jsonl
SUMMARY
model_name
Qwen/Qwen3-Coder-30B-A3B-Instruct              52.712228
claude-haiku-4-5-20251001                      58.810910
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct    45.081214
gemini-2.5-flash-lite                          81.765247
gpt-4o                                         64.756359
meta-llama/Llama-3.1-8B                        39.779344
Name: is_equivalent, dtype: float64
📊 Plots saved to /Users/obby/Documents/experiment/random/s

In [ ]:
# GENERATE MORE PLOTS AND TABLES
!python generate_detailed_plots.py

In [ ]:
# Cell 4: Backup to Drive
# Copies the entire local run folder to Google Drive

run_name = os.path.basename(latest_run)
drive_dest = f'{DRIVE_BASE}/runs/{run_name}'

print(f"🚀 Backing up {run_name} to Drive...")
if os.path.exists(drive_dest):
    shutil.rmtree(drive_dest)
shutil.copytree(latest_run, drive_dest)

print(f"✅ Backup Complete: {drive_dest}")